# Tiled assembly (template)

Single-mutant library for a CDS longer than one oligo: split into codon-aligned tiles, amplify each out of the pool with its own primer pair, and Golden Gate each into a destination vector holding the rest of the CDS. Fill in the input cell, then run top to bottom. A worked example is in `../tutorials/02-tiled-assembly.ipynb`.

In [ ]:
from library_designer import (
    LibrarySpec, CodonOptimizationParams, TiledAssemblyParams, SubstitutionScan,
)

# ===== EDIT =====
spec = LibrarySpec(
    name="my_tiled_library",
    protein_sequence="",              # your protein, one-letter, no stop codon
    substitutions=["A"],              # ["A"] alanine scan; list("ACDEFGHIKLMNPQRSTVWY") for full DMS
    cds=None,                         # native CDS to freeze verbatim; None to codon-optimize the protein
    optimization=CodonOptimizationParams(species="e_coli"),
    platform="twist_oligo_pools",
    tiled=TiledAssemblyParams(
        oligo_budget=300,             # hard cap on the final oligo length (bp)
        enzyme="BsaI",
        primer_set="subramanian2018", # bundled set, or a path to your own primer CSV
        optimize_overhangs=True,      # move the tile boundaries to the least similar overhangs
        pad_oligos=True,              # pad every oligo to one length (tiles differ in size)
        # pad_target=300,             # pad to this length instead of the longest oligo needed
        # starting_vector="my_backbone.gb",  # point at your destination plasmid for full vector maps
    ),
    seed=0,
)
spec

## Build, tile, and check

Each tile also gets a WT member of its own (`WT_Tile_0`, ...), the tile window straight from the reference, since a tile is amplified and assembled on its own and needs its own unmutated clone. Set `tiled.wt_controls = False` above to order only the mutants.

In [ ]:
lib = SubstitutionScan(spec).generate().codon_optimize().tile()
print(lib.summary())   # tiles, oligo lengths, and QC (junction sites, overhangs)

## Overhangs

Review these before ordering. Each tile is dropped in by a pair of four-base Golden Gate overhangs, read off the CDS at the tile boundaries rather than picked from an orthogonal set. If the two are too alike the cut vector re-closes empty or the fragment goes in backwards, so aim for at most one shared base.

The unit is one tile. It is amplified out of the pool on its own and assembled into the vector built around its own window, so two tiles' overhangs never meet and only a tile against itself can misfire.

The boundary is the only handle on those bases, which is what `optimize_overhangs` uses: the budget caps a tile, the balanced split sits under the cap, and the spare codons let the boundaries move. `pad_oligos` then evens the pool back out to one length, filling between each primer and the site next to it, outside what the enzyme releases.

In the matrix, only the graded cells matter: the boxed pairs, which are a tile's own two ends, and the diagonal, where a hot cell is a palindrome. Cross-tile cells are faded.

In [ ]:
lib.plot_overhangs()

pairs = lib.overhang_pairs()              # one row per tile; all_pairs=True adds the cross-tile rows
print(pairs["risk"].value_counts().to_dict())
pairs                                     # worst first; lib.overhangs() lists the ends themselves

## Export

In [ ]:
import os

lib.drop_failed()
out = lib.run_dir("out")                  # out/<name>_<date>_<time>, so a re-run keeps its own files

lib.to_oligo_pool(out / "oligos.csv")     # the pooled synthesis order (name, sequence)
lib.to_primer_order(out / "primers.csv")  # per-tile amplification primers (IDT bulk format)
lib.to_vectors(out / "vectors.csv")       # per-tile destination vectors (manifest)
lib.to_design_specs(out / f"{lib.spec.name}_design_specs.json")
# With a starting_vector set, also emit annotated plasmid maps:
# lib.to_vector_maps(out / "vectors")

print("wrote files to", os.path.abspath(out))
for root, _, files in os.walk(out):
    for f in sorted(files):
        print("  ", os.path.join(root, f))